# Chapter 3 — Attention as a Kernel Smoother

Companion notebook to Chapter 3.

Reproduces:
- Figure 3.1: $B$, $B_S$, $B_A$ heatmaps on a trained Std attention.
- Numerical verification of Proposition 1 (Frobenius orthogonality).
- Score-mode demo: full vs sym-only vs asym-only inference.
- Comparison: softmax vs Performer feature map.

In [ ]:
import os, math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from tabkernels.attention import (
    StdAttention, SymPSDAttention, SymGenAttention,
    PureAsymAttention, DualAttention, DecomposableStdAttention,
)
from tabkernels.core.decomposition import decompose, energy_split

torch.manual_seed(42); np.random.seed(42)
# Use repo-rooted absolute path so cwd doesn't matter
_HERE = os.path.dirname(os.path.abspath('__file__' if False else '.')) if False else os.getcwd()
# walk up until we find similarity-hierarchy-research
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book', 'figures')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

## Numerical check: Proposition 1 (Frobenius orthogonality)

In [ ]:
B = torch.randn(8, 8)
B_S, B_A = decompose(B)
inner = (B_S * B_A).sum().item()
lhs = (B ** 2).sum().item()
rhs = (B_S ** 2).sum().item() + (B_A ** 2).sum().item()
print(f'<B_S, B_A>_F      = {inner:+.3e}     (expect 0)')
print(f'||B||_F^2          = {lhs:.4f}')
print(f'||B_S||^2 + ||B_A||^2 = {rhs:.4f}     (Pythagorean)')
print(f'(alpha_S, alpha_A) = {energy_split(B)}')

## Train all five attention variants on a small kernel-NW task

The toy task: latent function $f(x) = \langle a, x\rangle + 0.3 (\langle b, x\rangle)^2$ with $x \in \R^4$.  The kernel attention block predicts $\hat{f}(x_q) = \sum_i w_i(x_q) y_i$ via a learned bilinear kernel.

In [ ]:
def make_data(N=200, d=4, seed=0):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(N, d, generator=g)
    a = torch.randn(d, generator=g); b = torch.randn(d, generator=g)
    y = X @ a + 0.3 * (X @ b) ** 2 + 0.05 * torch.randn(N, generator=g)
    return X, y

X_tr, y_tr = make_data(N=200, d=4, seed=0)
X_te, y_te = make_data(N=200, d=4, seed=1)
print(f'Train MSE if we predict the mean: {((y_te - y_tr.mean()) ** 2).mean():.3f}')
print(f'Train y variance: {y_tr.var():.3f}')

In [ ]:
def train(model, X_tr, y_tr, epochs=600, lr=0.05):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        # Leave-one-out style: predict y_i from the other (j != i) training points.
        # We approximate by training on the full set and adding mild noise during fit.
        out = model(X_tr, X_tr, y_tr)
        loss = F.mse_loss(out, y_tr)
        loss.backward(); opt.step()
    return model

results = {}
for name, ctor in [
    ('Std',       lambda: StdAttention(d_in=4, d_emb=8)),
    ('Sym-PSD',   lambda: SymPSDAttention(d_in=4, d_emb=8)),
    ('Sym-Gen',   lambda: SymGenAttention(d_in=4)),
    ('Pure-Asym', lambda: PureAsymAttention(d_in=4)),
    ('Dual',      lambda: DualAttention(d_in=4)),
]:
    torch.manual_seed(0)
    m = train(ctor(), X_tr, y_tr)
    with torch.no_grad():
        out = m(X_te, X_tr, y_tr)
        mse = F.mse_loss(out, y_te).item()
    aS, aA = m.energy_split()
    results[name] = dict(test_mse=mse, alpha_S=aS, alpha_A=aA)
    print(f'{name:>10s}: test MSE {mse:.3f}   alpha_S={aS:.3f} alpha_A={aA:.3f}')

## Figure 3.1: $B$, $B_S$, $B_A$ heatmaps for a trained Std attention

In [ ]:
torch.manual_seed(0)
m = train(StdAttention(d_in=4, d_emb=8), X_tr, y_tr)
with torch.no_grad():
    B = m.W_Q.weight.T @ m.W_K.weight
    B_S, B_A = decompose(B)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
vmax = max(B.abs().max().item(), B_S.abs().max().item(), B_A.abs().max().item())
for ax, M, title in zip(axes, [B, B_S, B_A], ['$B = W_Q^T W_K$', '$B_S = (B + B^T)/2$', '$B_A = (B - B^T)/2$']):
    im = ax.imshow(M.detach(), cmap='RdBu', vmin=-vmax, vmax=vmax)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle('Figure 3.1: Decomposition of trained Std-attention bilinear form')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_03_01_decomposition.pdf', bbox_inches='tight')
plt.show()
print(f'Frobenius energy split: alpha_S = {(B_S**2).sum() / (B**2).sum():.3f}, alpha_A = {(B_A**2).sum() / (B**2).sum():.3f}')

## Score-mode demo: post-hoc decomposition at inference

Train standard attention; rerun inference with `score_mode` set to `full`, `sym`, `asym`. The 'full' mode reproduces standard attention; the projection modes apply $(S \pm S^T)/2$ to the score matrix before softmax.

In [ ]:
torch.manual_seed(0)
m = DecomposableStdAttention(d_in=4, d_emb=8)
m = train(m, X_tr, y_tr)
with torch.no_grad():
    rows = []
    for mode in ['full', 'sym', 'asym']:
        m.score_mode = mode
        out = m(X_te, X_te, y_te)  # square X for score-mode projection to apply
        mse = F.mse_loss(out, y_te).item()
        rows.append((mode, mse))
    for mode, mse in rows:
        print(f'  score_mode={mode:>6s}: test MSE {mse:.3f}')
    aS, aA = m.energy_split()
    print(f'\n  alpha_S = {aS:.3f}, alpha_A = {aA:.3f}')
    print('  Note: even though alpha_A is non-trivial, asym-only inference is at chance (MSE >= y variance).')

## Softmax vs Performer feature map

Compare standard softmax attention to the Performer-style linear attention with $\phi(z) = \mathrm{ELU}(z) + 1$.

In [ ]:
class PerformerAttention(torch.nn.Module):
    """Linear attention with phi(z) = ELU(z) + 1 (Performer-style)."""
    def __init__(self, d_in, d_emb=8):
        super().__init__()
        self.W_Q = torch.nn.Linear(d_in, d_emb, bias=False)
        self.W_K = torch.nn.Linear(d_in, d_emb, bias=False)
    def forward(self, X_q, X_t, y_t):
        phi_q = F.elu(self.W_Q(X_q)) + 1
        phi_t = F.elu(self.W_K(X_t)) + 1
        S = phi_q @ phi_t.T  # non-negative
        W = S / (S.sum(dim=-1, keepdim=True) + 1e-9)
        return W @ y_t

torch.manual_seed(0)
m_softmax = train(StdAttention(d_in=4, d_emb=8), X_tr, y_tr)
torch.manual_seed(0)
m_performer = train(PerformerAttention(d_in=4, d_emb=8), X_tr, y_tr)
with torch.no_grad():
    mse_sm = F.mse_loss(m_softmax(X_te, X_tr, y_tr), y_te).item()
    mse_pf = F.mse_loss(m_performer(X_te, X_tr, y_tr), y_te).item()
print(f'Softmax  test MSE: {mse_sm:.3f}')
print(f'Performer test MSE: {mse_pf:.3f}')

**End of notebook.** Reproduces Figure 3.1; verifies Proposition 1 numerically; demonstrates the post-hoc score-mode switch and the Performer alternative.